# Section 7 — Practical 2: Local AI Governance & Approval Rules

**Windows JupyterLab + Ollama Qwen3 (no AWS, no paid API).**

This lab uses **fictional** store data and simulates approvals. It does not process payments, issue real refunds, or verify any manager's identity. The policy engine—not the AI model—decides whether a simulated action is permitted.

Run **Code Cell 1 to Code Cell 8** in order with **Shift + Enter**. The numbers refer to code cells, not Markdown headings.

## Cell 1 — Check Ollama and Qwen3

In [ ]:
import json, sys, urllib.request, urllib.error
from datetime import datetime, timezone
from pathlib import Path

OLLAMA_URL = "http://127.0.0.1:11434"
MODEL = "qwen3:1.7b"
with urllib.request.urlopen(f"{OLLAMA_URL}/api/tags", timeout=8) as resp:
    models = json.load(resp)
available = [m["name"] for m in models.get("models", [])]
assert MODEL in available, f"Missing {MODEL}. In PowerShell run: ollama pull {MODEL}"
print("Python:", sys.version.split()[0])
print("PASS: Local Ollama + Qwen3 available; AWS not required")

## Cell 2 — Define fictional policy and actions

In [ ]:
POLICY = {
    "currency": "INR",
    "small_refund_limit": 500,
    "maximum_simulated_refund": 5000,
    "allowed_inventory_ids": ["SKU-101", "SKU-102"],
}
INVENTORY = {"SKU-101": 12, "SKU-102": 0}
SAMPLE_CASES = [
    {"action":"inventory_lookup","item_id":"SKU-101"},
    {"action":"refund_request","amount":200,"approval_role":None},
    {"action":"refund_request","amount":200,"approval_role":"manager"},
    {"action":"refund_request","amount":2000,"approval_role":"supervisor"},
    {"action":"refund_request","amount":9000,"approval_role":"supervisor"},
    {"action":"issue_refund","amount":200},
]
print("PASS: Fictional inventory and approval policy loaded")

## Cell 3 — Build deterministic governance gate (never executes payments)

In [ ]:
def policy_gate(action, *, amount=None, approval_role=None, item_id=None):
    """Return a policy verdict. Never calls any payment, refund, or external API."""
    if action == "inventory_lookup":
        if item_id not in INVENTORY:
            return {"decision":"DENY", "reason":"Unknown item"}
        return {"decision":"ALLOW_READ_ONLY", "reason":"Fictional stock lookup", "quantity":INVENTORY[item_id]}
    if action != "refund_request":
        return {"decision":"DENY", "reason":"Action not allowed; no real financial actions"}
    if type(amount) not in (int, float) or not (0 < amount <= POLICY["maximum_simulated_refund"]):
        return {"decision":"DENY", "reason":"Invalid amount or exceeds limit"}
    needed = "manager" if amount <= POLICY["small_refund_limit"] else "supervisor"
    if approval_role != needed:
        return {"decision":"NEEDS_HUMAN_APPROVAL", "reason":f"Requires {needed} approval"}
    return {"decision":"ALLOW_SIMULATION_ONLY", "reason":f"Approval label {needed} supplied; simulated action only"}

print("PASS: Deterministic governance gate created")

## Cell 4 — Test allowed, approval-required and forbidden decisions

In [ ]:
assert policy_gate("inventory_lookup", item_id="SKU-101")["decision"] == "ALLOW_READ_ONLY"
assert policy_gate("refund_request", amount=200)["decision"] == "NEEDS_HUMAN_APPROVAL"
assert policy_gate("refund_request", amount=200, approval_role="manager")["decision"] == "ALLOW_SIMULATION_ONLY"
assert policy_gate("refund_request", amount=2000, approval_role="manager")["decision"] == "NEEDS_HUMAN_APPROVAL"
assert policy_gate("refund_request", amount=2000, approval_role="supervisor")["decision"] == "ALLOW_SIMULATION_ONLY"
assert policy_gate("refund_request", amount=9000, approval_role="supervisor")["decision"] == "DENY"
assert policy_gate("issue_refund", amount=200)["decision"] == "DENY"
assert policy_gate("export_customer_data")["decision"] == "DENY"
print("PASS: 8 governance decision tests passed")

## Cell 5 — Ask local Qwen3 to explain the policy (not decide approvals)

In [ ]:
import time

def local_qwen(prompt):
    last_error = None
    for tokens in (550, 900):
        payload = {
            "model": MODEL,
            "stream": False,
            "think": False,
            "messages": [
                {"role":"system","content":"Explain the provided fictional company policy in simple Hindi. Never claim to grant approval or execute refunds. Do not invent policy numbers."},
                {"role":"user","content":prompt},
            ],
            "options":{"num_predict":tokens,"temperature":0},
        }
        req = urllib.request.Request(f"{OLLAMA_URL}/api/chat", data=json.dumps(payload).encode("utf-8"), headers={"Content-Type":"application/json"}, method="POST")
        try:
            with urllib.request.urlopen(req, timeout=180) as resp:
                data = json.load(resp)
            content = data.get("message",{}).get("content", "").strip()
            if content:
                return content
            last_error = "Qwen3 returned empty content"
        except (urllib.error.URLError, TimeoutError, ValueError) as exc:
            last_error = str(exc)
    raise RuntimeError(f"Qwen3 did not return a final answer: {last_error}")

start = time.perf_counter()
AI_EXPLANATION = local_qwen("काल्पनिक स्टोर: ₹500 तक refund request के लिए manager label, ₹500 से ₹5,000 तक supervisor label चाहिए। ₹5,000 से अधिक deny। यह केवल simulation है। 4 वाक्यों में समझाओ।")
AI_LATENCY_SEC = round(time.perf_counter() - start, 2)
print("Qwen3 explanation (not an approval):", AI_EXPLANATION[:1200])
print("Response time (seconds):", AI_LATENCY_SEC)
print("PASS: Qwen3 gave a real local policy explanation")

## Cell 6 — Run simulated actions through the policy engine and keep an audit log

In [ ]:
AUDIT = []
for index, case in enumerate(SAMPLE_CASES, 1):
    decision = policy_gate(**case)
    record = {
        "event_no": index,
        "time_utc": datetime.now(timezone.utc).isoformat(),
        "action":case["action"],
        "amount":case.get("amount"),
        "item_id":case.get("item_id"),
        "approval_role_label":case.get("approval_role"),
        "decision":decision["decision"],
        "reason":decision["reason"],
        "is_real_transaction":False,
    }
    AUDIT.append(record)
    print(f"{index}. {record['action']}: {record['decision']}")
assert all(not r["is_real_transaction"] for r in AUDIT)
print("PASS: All fictional actions logged; no real transactions executed")

## Cell 7 — Final governance verification

In [ ]:
SUMMARY = {
    "section":7,
    "practical":"Local governance and approval rules",
    "model":MODEL,
    "api_cost_inr":0,
    "tests_passed":8,
    "audit_events":len(AUDIT),
    "decision_counts":{d:sum(r["decision"] == d for r in AUDIT) for d in sorted({r["decision"] for r in AUDIT})},
    "qwen3_response_seconds":AI_LATENCY_SEC,
    "financial_transactions_executed":0,
    "note":"Learning exercise; approval_role is only a label and is NOT an authenticated human approval.",
}
assert SUMMARY["audit_events"] == len(SAMPLE_CASES)
assert SUMMARY["financial_transactions_executed"] == 0
assert SUMMARY["decision_counts"]["DENY"] >= 2
assert SUMMARY["decision_counts"]["NEEDS_HUMAN_APPROVAL"] >= 1
print("PASS: Governance checks and audit summary validated")
print(json.dumps(SUMMARY, indent=2, ensure_ascii=False))

## Cell 8 — Save the report and audit data

In [ ]:
out_dir = Path.cwd()
json_path = out_dir / "section7_governance_audit.json"
md_path = out_dir / "section7_governance_report.md"
json_path.write_text(json.dumps({"summary":SUMMARY, "events":AUDIT}, indent=2, ensure_ascii=False), encoding="utf-8")
md_path.write_text(
    "# Section 7 — Local AI Governance Report\n\n"
    + "**This is an offline, fictional simulation—not AWS Bedrock Guardrails or AgentCore Policy.**\n\n"
    + f"- Qwen3 model: {MODEL}\n- Policy tests: {SUMMARY['tests_passed']}\n"
    + f"- Audit events: {len(AUDIT)}\n- Real financial transactions: 0\n"
    + f"- Qwen3 response time: {AI_LATENCY_SEC} seconds\n\n"
    + "## AI Explanation (not an approval)\n\n" + AI_EXPLANATION + "\n\n"
    + "## Governance limitation\n\nAn approval role label does not verify a human identity. Never connect this demo to real payment actions.\n",
    encoding="utf-8",
)
assert json_path.exists() and md_path.exists()
print("Saved:", json_path.name)
print("Saved:", md_path.name)
print("PASS: Section 7 Governance Practical completed and reports saved")

## Completion criteria

All 8 code cells must PASS, including the real Qwen3 reply in Cell 5 and both output files in Cell 8. This is a **local governance demonstration**. It does not replace the original paid AWS Bedrock Guardrails, AgentCore Policy (Cedar), governance checklist or human approval workflows.